In [ ]:
# bootstrap: Colab clone + local import of `distillab` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/distillation/distill-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "distillab").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 04 · A distilled draft for speculative decoding: acceptance is the student's metric

**Tier:** T1 on a rented 24 GB card (L4 or RTX 4090): target `Qwen/Qwen3-4B`, drafts `Qwen/Qwen3-0.6B` off the
shelf and a 0.6B distilled on the target's own outputs, compared through vLLM's `--speculative-config` and its
spec-decode counters (model ids, fits and flags verify). T0 (default): the tiny teacher of notebook 01 is the
target and its students are drafts. Their acceptance is computed from the models' own distributions with torch
on a CPU (about 50 seconds; the recorded run without torch). A bundled pair of `/metrics` scrapes, **synthetic**
at $\alpha = 0.7$, teaches vLLM's counters.

## The one-minute version

* **A draft model is a student graded on one number:** how often the target accepts what it proposes. Per position
  $\alpha = \sum_v \min(p(v), q(v)) = 1 - \mathrm{TV}(p, q)$. One target pass then yields
  $(1 - \alpha^{k+1}) / (1 - \alpha)$ tokens, and the speedup is that over $(k\,c + 1)$, with $c$ the cost of a
  draft step (serving-engine PRIMER §7, whose formulas `minengine.spec` implements; PRIMER §7 "A distilled draft for
  speculative decoding").
* **Distilling the draft on the target's outputs raises $\alpha$.** An off-the-shelf small model of the same family
  was trained on other data, and that distribution mismatch lowers $\alpha$. For a draft, do *not* filter by a
  verifier: the goal is to match the target, mistakes included.
* **Read vLLM's numbers carefully.** vLLM 0.30.0 drafts *greedily* by default, so the acceptance is
  $p(\operatorname{argmax} q)$, not $\sum \min(p, q)$. Its "acceptance rate" is
  $\text{accepted} / \text{drafted} = (E - 1)/k$, not $\alpha$. $\alpha$ is the per-position rate at position 0, and
  the mean acceptance length $1 + \text{accepted}/\text{drafts}$ is $E$.
* **Same vocabulary or nothing:** vLLM's `draft_model` method compares `vocab_size`. Qwen3-0.6B → Qwen3-4B passes
  (151,936 both), and Qwen2.5-0.5B → Qwen2.5-7B fails (151,936 against 152,064).

In [ ]:
import math, shlex, statistics
from distillab import draft as DR, env, metrics as M
from distillab.cost import load_config, shape
from distillab.report import table
from distillab.tinylm.curves import label, load_experiments, load_recorded

print(env.describe())
P_PRIMER, Q_PRIMER = [0.5, 0.3, 0.15, 0.05], [0.2, 0.2, 0.2, 0.4]
a = DR.acceptance_rate(P_PRIMER, Q_PRIMER)
print(f"serving-engine PRIMER §7's example: alpha = {a:.2f}; a greedy draft (argmax q = token 3) is accepted with p(3) = "
      f"{DR.greedy_acceptance(P_PRIMER, Q_PRIMER):.2f}")
print(table([{"k": k, **{f"E at alpha {al}": round(DR.expected_tokens(al, k), 3) for al in (0.6, 0.8, 0.95)}} for k in (1, 2, 4, 6, 8)],
            title="Tokens per target pass, (1 - alpha^(k+1)) / (1 - alpha)"))

## Worked example: vLLM's spec-decode counters

vLLM exports four counters: drafts (verify steps that had drafts), draft tokens, accepted tokens, and accepted
tokens per position (label `position` = $0 \ldots k - 1$). Position $i$ counts only when positions $0 \ldots i$ were
all accepted. The bundled scrapes are **synthetic**: 20,000 drafts of $k$ = 4 at $\alpha = 0.7$, made by
`DR.simulate_counters`, to show how the counters read and not what any draft achieves.

In [ ]:
from importlib import resources
text = lambda n: resources.files("distillab.assets").joinpath("samples", n).read_text()  # noqa: E731
BEFORE, AFTER = M.parse(text("spec_decode_metrics_before.txt")), M.parse(text("spec_decode_metrics_after.txt"))
print(text("spec_decode_metrics_after.txt"))

## Exercise 4.1 — from counters to α

Write `read_counters(before, after)` returning `(alpha, mean_length, rate)`. `alpha` is the per-position rate at
position 0 (accepted at position 0 over drafts). `mean_length` is $1 + \text{accepted} / \text{drafts}$ (vLLM counts
the target's bonus token). `rate` is accepted / draft tokens, the number vLLM logs as "Avg Draft acceptance rate".
Use `after.value(name)` and `before.value(name)` with the names in `distillab.metrics`, and
`after.by_label(M.SPEC_ACCEPTED_PER_POS, "position")` for the per-position counts.

In [ ]:
def read_counters(before, after) -> tuple:
    d = lambda n: after.value(n) - before.value(n)  # noqa: E731
    drafts, draft_tokens, accepted = d(M.SPEC_DRAFTS), d(M.SPEC_DRAFT_TOKENS), d(M.SPEC_ACCEPTED)
    pos0 = after.by_label(M.SPEC_ACCEPTED_PER_POS, "position")["0"] - before.by_label(M.SPEC_ACCEPTED_PER_POS, "position").get("0", 0.0)
    return pos0 / drafts, 1 + accepted / drafts, accepted / draft_tokens

In [ ]:
alpha, mean_len, rate = read_counters(BEFORE, AFTER)
ref = M.spec_decode(BEFORE, AFTER)
assert abs(alpha - ref["alpha_pos0"]) < 1e-12 and abs(mean_len - ref["mean_acceptance_length"]) < 1e-12
assert abs(rate - ref["acceptance_rate"]) < 1e-12
v = DR.vllm_views(alpha, 4)
assert abs(mean_len - v["mean_acceptance_length"]) < 0.05 and abs(rate - (mean_len - 1) / 4) < 1e-12
print(f"✅ [synthetic counters] alpha {alpha:.3f}, mean acceptance length {mean_len:.3f} (formula at that alpha, k = 4: "
      f"{v['mean_acceptance_length']:.3f}), vLLM's 'acceptance rate' {rate:.3f}, which is (E - 1)/k, not alpha")

## Worked example: the tiny drafts

The target is notebook 01's tiny teacher. Three students are drafts: **off-the-shelf** (`hard`, trained on the
labelled set rather than the target's outputs), **distilled** (`seqkd_all`, SFT on the target's own samples,
unfiltered), and **distilled and verified** (`seqkd`, the same with the verifier filter). For each position of the
target's own samples, $\alpha = \sum \min(p, q)$ and the greedy acceptance $p(\operatorname{argmax} q)$. About 50
seconds with torch.

In [ ]:
if env.has_torch():
    from distillab.tinylm import train as T
    RUN = T.run(T.DistillConfig(methods=("hard", "seqkd_all", "seqkd")), log=lambda *a: None, keep_models=True)
    MODELS = RUN["models"]
    POS = {m: DR.tiny_acceptance(MODELS[m], MODELS["teacher"], RUN["teacher_seqs"]) for m in ("hard", "seqkd_all", "seqkd")}
    ALPHA = {m: RUN["students"][m]["final"]["accept"] for m in POS}
    GREEDY = {m: RUN["students"][m]["final"]["accept_greedy"] for m in POS}
    ALPHA["target itself"] = 1.0
    GREEDY["target itself"] = RUN["teacher"]["eval"]["accept_greedy"]
    LABEL = label(RUN)
    print(table([{"position": i, **{m: POS[m][i]["alpha"] for m in POS}} for i in range(len(POS["hard"]))],
                title=f"[{LABEL}] alpha by completion position"))
else:
    RUN, EXP = load_recorded(), load_experiments()
    LABEL = RUN["source"]
    finals = {"hard": RUN["students"]["hard"]["final"], "seqkd_all": EXP["seqkd_all"]["final"], "seqkd": RUN["students"]["seqkd"]["final"]}
    ALPHA = {m: f["accept"] for m, f in finals.items()}
    GREEDY = {m: f["accept_greedy"] for m, f in finals.items()}
    ALPHA["target itself"], GREEDY["target itself"] = 1.0, RUN["teacher"]["eval"]["accept_greedy"]
C_TINY = RUN["params"]["student"] / RUN["params"]["teacher"]
print(table([{"draft": m, "alpha (sampled draft)": ALPHA[m], "greedy draft": GREEDY[m]} for m in ALPHA],
            title=f"[{LABEL}] mean acceptance over the target's samples; draft/target size c = {C_TINY:.3f}"))

## Exercise 4.2 — the best draft and k

Given each draft's $\alpha$ and cost ratio $c$, write `best_draft(alphas, c, k_max=8)`: the `(name, k, speedup)`
with the highest `DR.speedup(alpha, k, c)` over every draft and every $k$ from 1 to `k_max`. Leave out
`"target itself"`, which is not a draft you can afford.

In [ ]:
def best_draft(alphas: dict, c: float, k_max: int = 8) -> tuple:
    cands = [(name, k, DR.speedup(al, k, c)) for name, al in alphas.items() if name != "target itself"
             for k in range(1, k_max + 1)]
    return max(cands, key=lambda x: x[2])

In [ ]:
name, k, sp = best_draft(ALPHA, C_TINY)
brute = max(((n, kk, DR.speedup(al, kk, C_TINY)) for n, al in ALPHA.items() if n != "target itself" for kk in range(1, 9)),
            key=lambda x: x[2])
assert (name, k) == brute[:2] and abs(sp - brute[2]) < 1e-12
assert best_draft({"x": 0.9, "y": 0.5}, 0.1)[0] == "x"
rows = [{"draft": n, "alpha": ALPHA[n], "best k": DR.best_k(ALPHA[n], C_TINY, 8),
         "speedup": round(DR.speedup(ALPHA[n], DR.best_k(ALPHA[n], C_TINY, 8), C_TINY), 3)} for n in ALPHA if n != "target itself"]
print(table(rows, title=f"[{LABEL}] speedup at the best k (c = {C_TINY:.3f})"))
print(f"✅ best: {name} at k = {k}, {sp:.2f}x. " + ("The drafts distilled on the target's outputs beat the off-the-shelf one."
      if min(ALPHA["seqkd_all"], ALPHA["seqkd"]) > ALPHA["hard"] else "In this run the off-the-shelf draft was not worst: read the alpha table."))

Two things to notice. In the recorded run the **unfiltered** distilled draft has the higher $\alpha$ (one run: check
the table above for yours). That is the expected direction: the verifier filter trained the other one toward
*better* answers than the target gives, which is the wrong goal for a draft. And the tiny $c$ (a quarter of the
target) makes long $k$ expensive. A real 0.6B draft for a 4B target is $c \approx 0.15$ by weight bytes, so its best
$k$ is larger (Exercise 4.4).

## Exercise 4.3 — greedy drafting

vLLM 0.30.0's default `draft_sample_method` is `"greedy"`: the draft proposes $\operatorname{argmax} q$, and the
target keeps it with probability $p(\operatorname{argmax} q)$. Write `greedy_accept(p, q)`. Then compute `perfect`:
the greedy acceptance of a *perfect* draft ($q$ = $p$) for the primer's target $p$. The check prints the tiny
models' two columns side by side.

In [ ]:
def greedy_accept(p, q) -> float:
    return float(p[max(range(len(q)), key=lambda i: q[i])])

perfect = None
perfect = greedy_accept(P_PRIMER, P_PRIMER)

In [ ]:
assert greedy_accept(P_PRIMER, Q_PRIMER) == 0.05 and perfect == 0.5
assert greedy_accept([0, 1, 0, 0], [0.1, 0.6, 0.2, 0.1]) == 1.0          # a greedy target (T = 0): accept iff argmaxes match
print(table([{"draft": m, "alpha, sampled draft": ALPHA[m], "greedy draft": GREEDY[m], "lost to greedy": round(ALPHA[m] - GREEDY[m], 4)}
             for m in ALPHA], title=f"[{LABEL}] what greedy drafting costs when the target samples at T = 1"))
print("✅ with a sampling target even a perfect draft is accepted only p(argmax p) of the time under greedy drafting; "
      "'draft_sample_method': 'probabilistic' uses Sigma min(p, q) (vLLM v0.30.0; verify)")

## Exercise 4.4 — can vLLM run this pair, and at what k?

Write `plan_draft(target_cfg, draft_cfg, alpha)`: return `None` when `DR.check_vocab` rejects the pair; otherwise
`(k, speedup)` at the best $k$, with $c$ the ratio of the two models' parameter counts (`shape(name).params()`, a
memory-bound decode step streams the weights). The check runs it for two pairs at $\alpha = 0.7$.

In [ ]:
def plan_draft(target: str, draft: str, alpha: float):
    ok, _ = DR.check_vocab(load_config(target), load_config(draft))
    if not ok:
        return None
    c = DR.cost_ratio(shape(draft).params(), shape(target).params())
    k = DR.best_k(alpha, c)
    return k, DR.speedup(alpha, k, c)

In [ ]:
assert plan_draft("qwen2.5-7b-instruct", "qwen2.5-0.5b-instruct", 0.7) is None
k4, sp4 = plan_draft("qwen3-4b", "qwen3-0.6b", 0.7)
c = shape("qwen3-0.6b").params() / shape("qwen3-4b").params()
assert (k4, sp4) == (DR.best_k(0.7, c), DR.speedup(0.7, DR.best_k(0.7, c), c))
rows = [{"alpha": al, "best k": plan_draft("qwen3-4b", "qwen3-0.6b", al)[0],
         "speedup (predicted)": round(plan_draft("qwen3-4b", "qwen3-0.6b", al)[1], 2)} for al in (0.5, 0.6, 0.7, 0.8, 0.9)]
print(table(rows, title=f"PREDICTED: Qwen3-0.6B drafting for Qwen3-4B, c = {c:.3f} (memory-bound, batch 1)"))
print("✅ Qwen2.5-0.5B cannot draft for Qwen2.5-7B (vocab_size 151,936 vs 152,064); Qwen3-0.6B -> 4B can. Each 0.1 of alpha "
      "a distilled draft gains moves the speedup more than any k does")

The speedups are an upper bound for batch 1. The formula assumes a memory-bound target step, and verification
of $k$ + 1 tokens stays about as cheap as one only while that holds. Larger batches make the target
compute-bound, and the gain fades (the 04 serving lab's notebook 05 measures where).

## On a real GPU (T1)

On a 24 GB card: serve the target with each draft in turn, run the same load, and read the counters before and
after with `read_counters`. `DR.speculative_config` builds and checks the JSON:

In [ ]:
for draft in ("Qwen/Qwen3-0.6B", "_run_outputs/draft-distilled"):
    spec = DR.speculative_config(draft, 4, draft_sample_method="probabilistic")
    print(shlex.join(DR.serve_args("Qwen/Qwen3-4B", spec, max_model_len=8192, gpu_memory_utilization=0.9)))
if env.server_url():
    live = M.spec_decode(M.parse(""), M.scrape(env.server_url(), headers=env.auth_headers()))
    print("DISTILLAB_URL:", "spec-decode counters since start:" if live.get("drafts") else "no spec-decode counters (no draft configured)",
          {k: round(v, 4) if isinstance(v, float) else v for k, v in live.items() if k != "per_position"})

To make the distilled draft, serve the *target* first and collect its answers without a verifier filter or
deduplication, every finished answer, right or wrong:

    python -m distillab teacher-data --url http://127.0.0.1:8000 --keep all --out _run_outputs/target

The lab's generated problems stand in for your traffic here; a real draft should see prompts like the ones it
will serve. Then SFT Qwen3-0.6B on them and serve the result as the draft:

    python -m distillab.hf.sft --model Qwen/Qwen3-0.6B --data _run_outputs/target_pc.jsonl --out _run_outputs/draft-distilled
    DRAFT=_run_outputs/draft-distilled deploy/any-gpu/serve_with_draft.sh
EAGLE-3 heads are the same idea at the feature level: a small head trained with a soft-target
cross-entropy to the frozen target's distribution, from the target's hidden states (the `eagle` repository's
training code; SpecForge trains them for vLLM and SGLang). Their speedups are quoted for 13B targets on 2×
RTX 3090 (verify before quoting them for your setup).

## In a design review

**Two minutes:** "A draft model is a student whose only metric is acceptance: $\alpha = \sum \min(p, q)$, and the
speedup follows from $\alpha$, $k$ and the draft's relative cost. We distil the draft on the target's own outputs,
not filtered by correctness, because its job is to predict the target, mistakes included. On our toy that raised
$\alpha$ over an off-the-shelf model trained on other data. We check vLLM's counters the right way. The mean
acceptance length is the speedup's numerator, the logged 'acceptance rate' is $\left(E - 1\right)/k$, and $\alpha$
is position 0's rate. Greedy drafting, vLLM's default, caps acceptance at $p(\operatorname{argmax} q)$ for a
sampling target. The pair must share `vocab_size`: Qwen3-0.6B drafting for Qwen3-4B works, and Qwen2.5-0.5B for
Qwen2.5-7B does not."

**Drill 1.** *vLLM says the acceptance rate is 40%. Is speculation working?* Maybe well: with $k$ = 4 that is a mean
acceptance length of 2.6, so 2.6 tokens per target pass. Judge by measured inter-token latency at your batch
size, not by that rate.

**Drill 2.** *Should the draft be trained on verified outputs?* No. Verification makes it a better model and a
worse predictor of the target. Train it on the target's own samples, at the temperature you serve.

**Drill 3.** *The distilled draft doubled $\alpha$ at batch 1, but throughput at batch 64 fell. Why?* At batch 64 the
target step is compute-bound: verifying $k$ + 1 tokens per sequence costs up to $k$ + 1 times as much, and the extra
tokens do not pay for it. Speculation is a latency tool. Enable it by load, as `num_speculative_tokens_per_batch_size`
allows in vLLM (verify).